# Demo 06 : Tracing with LangSmith

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## a · trainer

We enable tracing inside this trainer demo only. The endpoint comes from the EU-region `.env` setting, and the surrounding environment keeps its default tracing setting.

In [2]:
import json
import re
import sqlite3
from pathlib import Path

from langchain.tools import tool

DB = data("game-studio/studio.db")
SECTIONS = []
for path in sorted(Path(data_folder("game-studio/docs")).glob("*.md")):
    title, *parts = path.read_text(encoding="utf-8").split("\n## ")
    SECTIONS += [f"{title.splitlines()[0]}\n## {part}" for part in parts]


def rows(sql, *args):
    with sqlite3.connect(DB) as conn:
        conn.row_factory = sqlite3.Row
        return [dict(row) for row in conn.execute(sql, args)]


def words(text):
    return set(re.findall(r"[a-z0-9-]{3,}", text.lower()))


def best(texts, query, k):
    q = words(query)
    return sorted(texts, key=lambda t: -len(q & words(t)))[:k]

@tool
def game_info(title: str) -> str:
    """Look up a Comet Arcade game by its title or id. Returns id, title, genre, status and pitch."""
    t = title.strip()
    found = rows("SELECT game_id, title, genre, status, pitch FROM games WHERE game_id = ? OR title LIKE ?", t.upper(), f"%{t}%")
    return json.dumps(found[0]) if found else "Unknown game"

@tool
def assets_for_game(game_id: str) -> str:
    """List sprites, palettes and sounds for a game id such as CS-01, with colour and status."""
    return json.dumps(rows("SELECT asset_id, kind, name, colour, status FROM assets WHERE game_id = ?", game_id.strip().upper()))

@tool
def search_design_docs(query: str) -> str:
    """Search the design docs and the studio style guide. Returns the three best matching sections."""
    return "\n\n---\n\n".join(best(SECTIONS, query, 3))

tools = [game_info, assets_for_game, search_design_docs]
question = {"messages": [("human", "Playtesters lose Comet Snake to comets they cannot see. Look up the game first, then its assets, "
                                   "then the style guide section on the rule that applies. Which asset is at fault, and what should the Artist change?")]}


def show(result):
    for message in result["messages"]:
        print(message.type, str(getattr(message, "tool_calls", None) or message.content)[:150].replace("\n", " "))
    print("model calls:", sum(m.type == "ai" for m in result["messages"]))
    print("input tokens per call:", [m.usage_metadata["input_tokens"] for m in result["messages"] if m.type == "ai"])

In [3]:
import os
import time
from uuid import uuid4
from langchain.agents import create_agent
from langsmith import Client, trace, tracing_context

PROJECT = "ARI520-module04-building-agents"
print("project:", PROJECT)
print("endpoint:", os.getenv("LANGSMITH_ENDPOINT", "not configured"))
print("default tracing:", os.getenv("LANGSMITH_TRACING", "not configured"))
agent = create_agent(llm, tools=tools,
                     system_prompt="You are the studio assistant of Comet Arcade. Use the tools. Answer in three sentences.")
out = None
try:
    if not os.getenv("LANGSMITH_API_KEY") or not os.getenv("LANGSMITH_ENDPOINT"):
        raise RuntimeError("LangSmith key or EU endpoint is missing")
    client = Client(api_url=os.environ["LANGSMITH_ENDPOINT"],
                    api_key=os.environ["LANGSMITH_API_KEY"], timeout_ms=10000)
    list(client.list_projects(limit=1))
    with tracing_context(enabled=True, client=client, project_name=PROJECT):
        with trace("comet-contrast", run_type="chain", inputs=question,
                   run_id=uuid4()) as root_run:
            out = agent.invoke(question, {"recursion_limit": 16})
            root_run.end(outputs={"answer": out["messages"][-1].content})
    client.flush()
    project = next(client.list_projects(name=PROJECT, limit=1))
    for attempt in range(3):
        try:
            recorded = await client.runs.retrieve(str(root_run.id), project_id=str(project.id))
            break
        except Exception:
            if attempt == 2:
                raise
            time.sleep(0.5)
    print("trace recorded:", recorded.id)
except Exception as exc:
    print("tracing failed:", type(exc).__name__)
if out is None:
    with tracing_context(enabled=False):
        out = agent.invoke(question, {"recursion_limit": 16})
print("answer:", out["messages"][-1].content)
print("default tracing after:", os.getenv("LANGSMITH_TRACING", "not configured"))

project: ARI520-module04-building-agents
endpoint: https://eu.api.smith.langchain.com
default tracing: false


trace recorded: 6cecdff4-2cbc-489e-aae5-6e55d4deafd5
answer: Comet Snake’s comet sprite, **CS-SPR-03**, is at fault: its draft colour `#3a4a7a` is too close to the deep-space background, making the hazard hard to see. The style guide says game-ending hazards need at least 4.5:1 contrast and specifies the danger colour `#ff6b5e`. The Artist should keep the comet’s shape and change its colour to `#ff6b5e`.
default tracing after: false


We inspect the named project in LangSmith for the model calls, tool arguments, results and timing. If recording fails, the agent still answers and the cell prints the failure type.

## Things to try

Optional : we can change the game, the asset or the question and run the example again.